In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "PurpleBeam"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
     html.Center([
     html.Img(
        src='data:image/png;base64,{}'.format(
             encoded_image.decode()
        ),
        style={'width': '200px'}        ),
        html.H3("Developed by Kaseem Wright")
    ]),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),
    html.Div([
    html.Label("Select Rescue Type:"),

    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'All Animals (Reset)', 'value': 'Reset'},
            {'label': 'Water Rescue', 'value': 'Water'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
            {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'}
        ],
        value='Reset',
        labelStyle={'display': 'inline-block', 'marginRight': '20px'}
    )
]),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
    editable=False,
    filter_action="native",
    sort_action="native",
    sort_mode="multi",
    row_selectable="single",
    selected_rows=[0],
    page_action="native",
    page_current=0,
    page_size=10, 

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(
    className='row',
    style={
        'display': 'flex',
        'flexWrap': 'wrap',
        'width': '100%'
    },
    children=[
        html.Div(
            id='graph-id',
            style={
                'width': '50%',
                'minWidth': '400px',
                'boxSizing': 'border-box'
            }
        ),
        html.Div(
            id='map-id',
            style={
                'width': '50%',
                'minWidth': '400px',
                'boxSizing': 'border-box'
            }
        )
    ]
)
])

#############################################
# Interaction Between Components / Controller
#############################################




@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):

    if filter_type == 'Reset':
        query = {}

    elif filter_type == 'Water':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$regex': 'Labrador Retriever|Chesapeake Bay Retriever|Newfoundland',
                '$options': 'i'
            },
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }

    elif filter_type == 'Mountain':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$regex': 'German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler',
                '$options': 'i'
            },
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }

    elif filter_type == 'Disaster':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$regex': 'Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler',
                '$options': 'i'
            },
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        }

    else:
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))

    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')

@app.callback(
    Output('graph-id', 'children'),
    Input('datatable-id', 'derived_virtual_data')
)
def update_graphs(viewData):

    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame(viewData)

    if dff.empty or 'breed' not in dff.columns:
        return [html.P("No animal data available")]

    breed_counts = (
        dff['breed']
        .value_counts()
        .head(10)
        .reset_index()
    )

    breed_counts.columns = ['breed', 'count']

    fig = px.pie(
        breed_counts,
        names='breed',
        values='count',
        title='Top 10 Animal Breeds'
    )

    return [
        dcc.Graph(figure=fig)
    ]

@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'selected_columns')
)
def update_styles(selected_columns):
    if not selected_columns:
        return []

    return [
        {
            'if': {'column_id': column},
            'backgroundColor': '#D2F3FF'
        }
        for column in selected_columns
    ]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable

@app.callback(
    Output('map-id', 'children'),
    [
        Input('datatable-id', 'derived_virtual_data'),
        Input('datatable-id', 'derived_virtual_selected_rows')
    ]
)
def update_map(viewData, index):

    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame(viewData)

    if dff.empty:
        return [html.P("No location data available")]

    row = index[0] if index else 0

    if row >= len(dff):
        row = 0

    animal = dff.iloc[row]

    latitude = animal.get('location_lat')
    longitude = animal.get('location_long')

    if pd.isna(latitude) or pd.isna(longitude):
        return [html.P("Location unavailable for this animal")]

    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[latitude, longitude],
            zoom=10,
            children=[
                dl.TileLayer(),
                dl.Marker(
                    position=[latitude, longitude],
                    children=[
                        dl.Tooltip(str(animal.get('breed', 'Unknown'))),
                        dl.Popup([
                            html.H4("Animal Name"),
                            html.P(str(animal.get('name', 'Unknown')))
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

In [ ]:
print(df[['location_lat', 'location_long']].head())

/home/codio/.pyenv/versions/3.11.2/lib/python3.11/site-packages/jupyter_dash/comms.py:100: RuntimeWarning: coroutine 'Kernel.execute_request' was never awaited
  kernel.execute_request(stream, ident, parent)


Dash app running on https://fionagondola-lilynavy-3000.codio.io/proxy/8050/
